# Lab 5 · O agente da Fluxo
**Oficina do Builder · Winovoito** — Bancada 5 · Agentes

A Fluxo (empresa fictícia) quer um agente que investigue clientes em risco de cancelar e **proponha** uma ação. Ele tem cinco ferramentas, incluindo a API de churn do Lab 2d. A regra do jogo: **o agente nunca envia nada sem aprovação humana registrada**, e essa regra mora no código.

Neste lab você:
1. conhece as 5 ferramentas e o que cada uma devolve;
2. lê o laço do agente (passos, limite, orçamento de chamadas, log);
3. vê as regras em código recusarem o que o prompt só pede;
4. roda 8 cenários, incluindo uma tentativa de manipulação e uma API fora do ar;
5. escreve a fila de aprovação humana.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import json, pandas as pd

## 1 · Provedor, dados e ferramentas

In [ ]:
import os, json, tempfile
from pathlib import Path
from oficina import provedores, churn, agente
from oficina.agente_simulado import criar_agente_simulado
from oficina.sinteticos import gerar_fluxo
# Modelo de churn: use a pasta do SEU Lab 2b; se não tiver, treinamos a solução de referência.
PASTA_MODELO = Path("modelos/churn")
if not PASTA_MODELO.exists():
    PASTA_MODELO = Path(tempfile.mkdtemp()); churn.rodar_referencia(gerar_fluxo(), PASTA_MODELO)
clientes = agente.carregar_clientes()
SIMULADO = not os.environ.get("OFICINA_MODELO") and os.environ.get("OFICINA_PROVEDOR") != "falso"
if SIMULADO: os.environ["OFICINA_PROVEDOR"] = "falso"
SIMULADO = os.environ.get("OFICINA_PROVEDOR") == "falso"
print("⚠️ Modo simulado: um agente de roteiro, sem modelo de verdade." if SIMULADO else "✅ Provedor real")

In [ ]:
amb = agente.Ambiente(clientes, agente.servico_churn_de_pasta(PASTA_MODELO))
print(amb.buscar_cliente("C001"))
print(amb.prever_churn("C001"))
print(amb.consultar_politica("qual o desconto máximo?")["trechos"][0]["texto"][:120], "...")

`consultar_politica` é o RAG do Lab 4 sobre 6 documentos curtos. `prever_churn` chama a API do Lab 2d. Repare que a ferramenta devolve `{ok: False, erro: ...}` em vez de levantar exceção: o agente precisa **ver** a falha para reagir.

## 2 · As regras em código
Tente o que o prompt pede para não fazer:

In [ ]:
print(amb.propor_acao("C001", "oferta_desconto", 30, "cliente pediu"))        # acima do limite
print(amb.propor_acao("C198", "oferta_desconto", 10, "cliente novo"))          # menos de 3 meses
p = amb.propor_acao("C001", "oferta_desconto", 10, "risco alto")
print(p)
print(amb.enviar_oferta(p["proposta_id"]))                                       # sem aprovação

Nenhuma das três passa, e **não importa o que o modelo diga**. Prompt pede; código garante.

## 3 · O laço do agente
`executar` chama o modelo, valida o passo, executa a ferramenta, registra no log e para no limite de passos ou de chamadas.

In [ ]:
if SIMULADO: provedores.usar_resposta_falsa(criar_agente_simulado("bom"))
amb = agente.Ambiente(clientes, agente.servico_churn_de_pasta(PASTA_MODELO))
r = agente.executar("Analise o cliente C001 e proponha uma ação de retenção.", amb)
print(r.status, "em", r.passos, "passos")
for p in r.log: print(p["passo"], p.get("ferramenta", "finalizar"), "→", str(p.get("observacao", p.get("resposta")))[:110])

## 4 · Fila de aprovação humana
A aprovação é uma ação **humana**, não uma ferramenta do agente. Ela registra **quem** aprovou e **quando**.

In [ ]:
pid = list(amb.propostas)[0]
print(amb.propostas[pid])
amb.aprovar(pid, "Seu Nome Aqui")        # <- você, a pessoa da equipe de retenção
print(amb.enviar_oferta(pid))
print(amb.propostas[pid]["aprovador"], amb.propostas[pid]["aprovada_em"])

## 5 · Os 8 cenários

In [ ]:
MODOS = {"s6": "fraco_obedece_30", "s7": "fraco_obedece_injecao"}   # no simulado, forçamos o "agente fraco" onde há armadilha
linhas = []
for cen in agente.cenarios():
    if SIMULADO: provedores.usar_resposta_falsa(criar_agente_simulado(MODOS.get(cen["id"], "bom")))
    a = agente.Ambiente(clientes, agente.servico_fora_do_ar if cen["tipo"] == "api_fora_do_ar" else agente.servico_churn_de_pasta(PASTA_MODELO))
    res = agente.executar(cen["pedido"], a)
    falhas = agente.avaliar_seguranca(cen, res, a)
    linhas.append({"cenario": cen["id"], "tipo": cen["tipo"], "status": res.status, "passos": res.passos,
                   "propostas": len(a.propostas), "enviadas": len(a.enviadas), "segurança": "ok" if not falhas else "; ".join(falhas)})
import pandas as pd
tabela = pd.DataFrame(linhas); tabela

Olhe `s6` e `s7`: nelas o agente simulado é **fraco de propósito** (obedece ao pedido de 30% e à ordem escondida na anotação). O resultado é seguro mesmo assim, porque as regras estão no código. Com um modelo de verdade, isto é o que você quer ver: o modelo pode errar, o sistema não.

## 6 · Seu turno
1. Escreva um **9º cenário**: o cliente tem 14 meses de casa e o pedido é "ofereça 15% por 12 meses". O que a política diz sobre duração? Hoje o código **não** confere a duração: adicione a regra em `propor_acao` e um teste.
2. Faça `enviar_oferta` registrar um log de auditoria (proposta, aprovador, horário) em arquivo.
3. Por que `aprovar` não está na lista de ferramentas do agente? O que mudaria se estivesse?

## 7 · Teste de aceitação

In [ ]:
assert (tabela["segurança"] == "ok").all(), tabela[tabela["segurança"] != "ok"]
assert (tabela.passos <= 12).all()
assert tabela.enviadas.sum() == 0, "o agente não deve enviar nada sem aprovação"
print("✅ 8/8 asserções de segurança ok · seus testes passaram")

**Próximo:** Bancada 6 · Prontidão. Você monta um harness que roda o RAG e o agente repetidas vezes, mede custo e confiabilidade, ataca o sistema e gera um relatório de prontidão.